# 🏛️ SGLang 架构总览 — RadixAttention 设计哲学

**本文目标**：建立 SGLang 系统的完整心智模型，理解它与 vLLM 的根本差异。

读完这篇你会理解：
- SGLang 的 "结构化生成" 设计哲学
- RadixAttention vs PagedAttention 的本质区别
- SGLang Runtime (SRT) 的系统架构
- 为什么 SGLang 在 Agent 和前缀复用场景有明显优势

## 1. SGLang 的核心设计理念

### 1.1 两个拳头产品

```
SGLang = SGLang DSL (前端) + SGLang Runtime (后端)

前端: SGLang DSL (Domain-Specific Language)
  让开发者用 Python 写 LLM 交互逻辑
  内置: 并行调用、分支、约束生成、多模态

后端: SGLang Runtime (SRT)
  高性能推理引擎
  核心创新: RadixAttention (token-level prefix caching)
  支持: continuous batching, constrained decoding, 多模态

对比 vLLM:
  vLLM = 推理引擎 (只有后端)
  SGLang = 推理引擎 + 编程框架
```

### 1.2 SGLang 的 "结构化生成" 哲学

```
传统 LLM 推理: 自由文本生成
  prompt → model.generate() → 自由文本 → 解析 (可能失败)

SGLang: 结构化约束生成
  prompt + JSON Schema → model.generate(constrained) → 保证合法的 JSON
  或
  prompt + Regex → model.generate(regex=...) → 保证匹配正则

为什么要结构化?
  Agent 调用: tool_call 必须是合法 JSON → constrained generation 100% 保证
  数据提取: 从文本中提取结构化信息 → 用 Regex 约束输出格式
  代码生成: 约束 indent、禁止某些 pattern → 减少语法错误
```

### 1.3 与 vLLM 的核心差异定位

| | vLLM | SGLang |
|---|---|---|
| 核心创新 | PagedAttention (显存管理) | RadixAttention (前缀共享) |
| 缓存粒度 | Block-level (16 tokens) | **Token-level** |
| 前缀共享 | APC (hash matching) | **Radix Tree (自动发现)** |
| 约束生成 | 有限支持 | **一等公民** (FSM-based) |
| 编程模型 | API Server only | **DSL + API Server** |
| 最适合 | 通用 API 服务 | Agent / 前缀复用 / 结构化输出

## 2. SGLang Runtime 架构

```
┌─────────────────────────────────────────────────────────┐
│                   SGLang Runtime (SRT)                    │
├─────────────────────────────────────────────────────────┤
│                                                           │
│  ┌─────────────────┐      ┌─────────────────┐            │
│  │  HTTP API Server │      │  SGLang DSL      │            │
│  │  (OpenAI compat) │      │  (Python frontend)│           │
│  └────────┬────────┘      └────────┬────────┘            │
│           │                        │                      │
│  ┌────────┴────────────────────────┴────────┐            │
│  │         Tokenizer / Detokenizer           │            │
│  └─────────────────────┬────────────────────┘            │
│                        │                                  │
│  ┌─────────────────────┴────────────────────┐            │
│  │            Scheduler                      │            │
│  │  - Radix Tree 感知的调度                   │            │
│  │  - LPM (Longest Prefix Match) 策略        │            │
│  │  - Continuous batching                   │            │
│  └─────────────────────┬────────────────────┘            │
│                        │                                  │
│  ┌─────────────────────┴────────────────────┐            │
│  │         Radix Tree Cache Manager          │            │
│  │  - Token-level KV Cache 管理              │            │
│  │  - 自动前缀发现和共享                      │            │
│  │  - 引用计数 + LRU 淘汰                    │            │
│  └─────────────────────┬────────────────────┘            │
│                        │                                  │
│  ┌─────────────────────┴────────────────────┐            │
│  │         Model Runner (GPU Worker)         │            │
│  │  - FlashAttention / FlashInfer backend    │            │
│  │  - 支持 CUDA graph                       │            │
│  │  - 支持 tensor parallel                  │            │
│  └──────────────────────────────────────────┘            │
│                                                           │
└─────────────────────────────────────────────────────────┘
```

### 2.1 SGLang DSL 的编程模型

```python
import sglang as sgl

# SGLang 的编程模型: 用 Python 写 LLM 交互

@sgl.function
def multi_turn_agent(s, question):
    s += sgl.system("You are a helpful assistant with tools.")
    s += sgl.user(question)
    s += sgl.assistant(sgl.gen("answer", max_tokens=100))

# 关键特性:
# 1. sgl.gen("name", ...): 命名生成点 (可以后续引用)
# 2. sgl.gen("json_output", regex=json_regex): 约束生成
# 3. sgl.select("choice", choices=[...]): 从选项中选择
# 4. sgl.image("img"): 多模态输入
# 5. fork/join: 并行生成多个分支
```

### 2.2 源码目录结构

```
sglang/
├── srt/                          # SGLang Runtime (核心)
│   ├── server.py                 # HTTP API Server
│   ├── managers/
│   │   ├── scheduler.py          # 调度器
│   │   ├── tokenizer_manager.py  # Tokenizer 管理
│   │   └── cache_controller.py   # Radix Tree 缓存控制
│   ├── model_executor/
│   │   └── model_runner.py       # GPU 模型执行
│   └── mem_cache/
│       ├── radix_cache.py        # Radix Tree 实现 ★
│       └── memory_pool.py        # 显存池管理
├── lang/                         # SGLang DSL
│   ├── api.py                    # DSL API
│   └── backend/                  # 各种后端 (SRT, OpenAI, etc.)
└── test/                         # 测试
```

## 3. RadixAttention 核心概念预览

### 3.1 Radix Tree = 压缩前缀树

```
7 个请求在 Radix Tree 中的样子:

请求 A: "You are a helpful assistant. Hello!"
请求 B: "You are a helpful assistant. Hi!"
请求 C: "You are a helpful assistant. How are you?"
请求 D: "You are a helpful coder. Write Python."
请求 E: "You are a helpful coder. Write Rust."
请求 F: "You are a helpful coder. Write Go."
请求 G: "I am a robot."

Radix Tree:
                        ┌──────────┐
                        │   Root    │
                        └────┬─────┘
                             │
              ┌──────────────┴──────────────┐
              │ "You are a helpful "        │  ← 4 个请求共享!
              └──────────────┬──────────────┘
                             │
               ┌─────────────┴─────────────┐
               │                           │
        ┌──────┴──────┐            ┌──────┴──────┐
        │ "assistant."│            │ "coder. "   │
        └──────┬──────┘            └──────┬──────┘
               │                         │
    ┌──────────┼──────────┐     ┌────────┼────────┐
    │          │          │     │        │        │
┌───┴───┐ ┌───┴───┐ ┌───┴───┐ ┌┴──┐  ┌──┴──┐  ┌─┴──┐
│"Hello!"│ │"Hi!"  │ │"How   │ │WP │  │Rust │  │Go  │
│        │ │       │ │ are.."│ │   │  │     │  │    │
└────────┘ └───────┘ └───────┘ └───┘  └─────┘  └────┘

  请求 A   请求 B    请求 C   请求 D  请求 E  请求 F

注意: "I am a robot." (请求 G) 没有共享前缀 → 独立分支
```

### 3.2 Token-level 粒度意味着什么

```
场景: 两个请求, 前缀的前 15 个 token 相同, 第 16 个不同

vLLM APC (block_size=16):
  Block 0 = token[0:16]
  请求 A token[0:16]  vs 请求 B token[0:16]
  → 第 16 个 token 不同 → hash 不同 → MISS
  → 浪费了前 15 个 token 的共享机会!

SGLang RadixAttention (token-level):
  Radix tree 中 token[0]→token[1]→...→token[14] 已存储
  请求 B 走到 token[14] → 发现 token[15] 分叉
  → token[0:15] 命中! (15 个 token 复用)
  → token[15] 创建新分支

  额外收益: 如果请求 C 的 token[0:20] 和请求 A 完全相同
  → 走请求 A 的路径 20 个 token → 20 个命中

结论: Token-level 粒度能捕获 block 边界附近的"部分匹配"
```